In [ ]:
class AMOSVQADataset(Dataset):
    def __init__(self, args, tokenizer, mode="train"):
        self.args = args
        self.data_root = args.data_root
        self.tokenizer = tokenizer
        self.mode = mode
        
        print("Arguments provided in 'args':")
        for key, value in vars(self.args).items():
            print(f"{key}: {value}")

        self.image_tokens = "<im_patch>" * args.proj_out_num

        assert len(args.json_path) == len(args.data_root), "You need to provide the image directory for every dataset's JSON."

        self.data_list = self._make_combined_json(args)

        print(f"Length dataset: {len(self.data_list)}")

        train_transform = mtf.Compose(
            [
                mtf.RandScaleIntensity(factors=0.1, prob=0.5),
                mtf.RandShiftIntensity(offsets=0.1, prob=0.5),
                mtf.ToTensor(dtype=torch.float),
                mtf.Resize(args.data_img_size),
            ]
        )

        val_transform = mtf.Compose(
                [
                    mtf.ToTensor(dtype=torch.float),
                    mtf.Resize(args.data_img_size),
                ]
            )
        set_track_meta(False)

        if mode == 'train' or mode == "train_val":
            self.transform = train_transform
        elif mode == 'validation':
            self.transform = val_transform
        elif 'test' in mode:
            self.transform = val_transform

    def _make_combined_json(self, args):
        paths = args.json_path
        image_paths = args.data_root

        if isinstance(paths, list):
            combined = []
            for idx, p in enumerate(paths):
                img_path = image_paths[idx] if isinstance(image_paths, list) else image_paths
                with open(p, 'r') as f:
                    data = json.load(f)
                    for item in data:
                        item['volume_path'] = img_path + os.sep + item['case_id']
                    combined.extend(data)
            json_file = combined
        else:
            img_path = image_paths[0] if isinstance(image_paths, list) else image_paths
            with open(paths, 'r') as f:
                data = json.load(f)
                for item in data:
                    item['volume_path'] = img_path + os.sep + item['case_id']
            json_file = data
        return json_file

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        max_tries = 10
        for _ in range(max_tries):
            try:
                data = self.data_list[idx]
                image_path = data["image"]
                if image_path.startswith('/'):
                    image_abs_path = image_path 
                else:
                    image_abs_path = os.path.join(self.data_root, image_path)

                image = read_image(image_abs_path)
                image = self.transform(image)

                vqa_path = data["vqa"]
                if vqa_path.startswith('/'):
                    vqa_abs_path = vqa_path
                else:
                    vqa_abs_path = os.path.join(self.data_root, vqa_path)
                
                if vqa_abs_path.endswith('.json'):
                    with open(vqa_abs_path) as f:
                        qs = json.load(f) # dict
                        vqa_data = random.choices(qs, k=1)[0]
                        options = vqa_data["options"]
                        question = vqa_data["question"]
                        choices = "Choices: A. {} B. {} C. {} D. {}".format(options["A"], options["B"], options["C"], options["D"])
                        question = question + ' ' + choices
                else:
                    print(f"text Error in __getitem__ at index {idx}, file suffix should be .txt or .json")

                conversation = [{  
                    "role": "system", "content": "You are an AI assistant acting as a radiologist tasked with answering a multiple choice question based on a CT scan."},
                    {"role": "user", "content": self.image_tokens + ' ' + question}]
                        # {"role": "user", "content": question}]
                question = self.tokenizer.apply_chat_template(conversation, tokenize=False)

                text_tensor = self.tokenizer(
                    question + ' ' + answer, max_length=self.args.max_length, truncation=True, padding="max_length", return_tensors="pt",
                )
                input_id = text_tensor["input_ids"][0]
                attention_mask = text_tensor["attention_mask"][0]

                valid_len = torch.sum(attention_mask)
                if valid_len < len(input_id):
                    input_id[valid_len] = self.tokenizer.eos_token_id

                question_tensor = self.tokenizer(
                    question, max_length=self.args.max_length, truncation=True, padding="max_length", return_tensors="pt"
                )

                question_len = torch.sum(question_tensor["attention_mask"][0])

                label = input_id.clone()
                label[:question_len] = -100
                if self.tokenizer.pad_token_id == self.tokenizer.eos_token_id:
                    label[label == self.tokenizer.pad_token_id] = -100
                    if valid_len < len(label):
                        label[valid_len] = self.tokenizer.eos_token_id
                else:
                    label[label == self.tokenizer.pad_token_id] = -100

                ret = {
                    'image': image,
                    'input_id': input_id,
                    'label': label,
                    'attention_mask': attention_mask,
                    'question': question,
                    'answer': answer,
                    'question_type': "Caption",
                }

                return ret
            except Exception as e:
                print(f"Error in __getitem__ at index {idx}: {e}, name: {self.data_list[idx]}")
                idx = random.randint(0, len(self.data_list) - 1)

/home/jma/anaconda3/envs/flare_task5/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,generated,gt,name
0,"liver: The liver is of normal size and shape, ...","liver: The liver is normal in size and shape, ...",amos_5001.npy
1,liver: Liver is of normal size and shape with ...,liver: Liver surface is smooth. Liver parenchy...,amos_0140.npy
2,"liver: The liver surface is smooth, with a coo...","liver: The liver is of normal size and shape, ...",amos_0144.npy
3,liver: Liver is normal in size and shape with ...,liver: Liver parenchyma shows no significant a...,amos_0128.npy
4,liver: Liver size and shape are normal. Patchy...,"liver: The position, size, and contour of the ...",amos_5021.npy


In [ ]:
import json
import re
from typing import Dict, Any

def score_row(gt_raw: Any, gen_raw: Any) -> Dict[str, float]:
    gt  = text_block_to_dict(gt_raw)
    gen = text_block_to_dict(gen_raw)

    print(gt)
    print(gen)

    print(gt.keys())
    print(gen.keys())

    scores = {}
    # Evaluate sections present in ground-truth
    for key, gt_val in gt.items():
        if key in gen:
            print("HI")
            # scores[key] = green(gt_val, gen[key])
        else:
            scores[key] = 0.0

    # Penalize extra sections hallucinated by the model
    for key, gen_val in gen.items():
        if key not in gt:
            scores[key] = 0.0

    return scores


all_scores = []

for idx, row in df.iterrows():
    row_scores = score_row(row["gt"], row["generated"])
    all_scores.append(row_scores)

    print(all_scores)
    

{'liver': 'The liver is normal in size and shape, with coordinated proportion of liver lobes. Multiple low-density cystic lesions are seen in the liver, the largest in the upper right lobe (S8), measuring approximately 14mm × 20mm, with clear borders and a CT value of about 2 HU, without obvious enhancement. Patchy and inhomogeneous low-density focus with slightly clear edge is seen in the left inner lobe near the liver fissure (S4), about 8mm × 16mm, with a CT value of about 6-26 HU, showing slightly progressive enhancement in the portal venous and delayed phases, with a delayed phase CT value of about 37 HU. A small calcified high-density focus is seen at the edge of the right posterior lobe (S6).', 'biliary system': 'The intrahepatic duct system and common bile duct are not dilated and normal in course. The gallbladder is not enlarged, the wall is not thickened, and no abnormal density shadows are seen inside. No obvious abnormal enhancement is seen in the gallbladder during the enh

In [ ]:
import json
from pathlib import Path

# Paths
metadata_path = Path("/home/jma/datasets/mohammed/FLARE-Task5-MLLM-3D/train/archive/CT-RATE-Tr.json")         # Full metadata JSON file
volume_dir = Path("/home/jma/datasets/mohammed/FLARE-Task5-MLLM-3D/train/CT-RATE-2000/")          # Directory with the 2000 volume files
output_path = Path("/home/jma/datasets/mohammed/FLARE-Task5-MLLM-3D/train/CT-RATE-Tr.json")       # Where to save the filtered list

# Get all filenames in the 2K directory
selected_ids = {f.name for f in volume_dir.glob("*") if f.is_file()}

# Load full metadata
with open(metadata_path) as f:
    metadata = json.load(f)

# Filter entries
filtered = [item for item in metadata if item["case_id"] in selected_ids]

# Save the result
with open(output_path, "w") as f:
    json.dump(filtered, f, indent=2)

print(f"Filtered down to {len(filtered)} cases from {len(metadata)} entries.")


In [ ]:
import os
import shutil
from pathlib import Path

# Change this to your actual path
SOURCE_DIR = Path("/home/jma/datasets/mohammed/FLARE-Task5-MLLM-3D/train/CT-RATE-4791")
DEST_DIR = SOURCE_DIR.parent / "CT-RATE-2000"
DEST_DIR.mkdir(exist_ok=True)

# List all volume files (add more extensions if needed)
volume_extensions = [".nii", ".nii.gz"]
volumes = [f for f in SOURCE_DIR.iterdir() if f.is_file() and f.suffix in volume_extensions or f.name.endswith(".nii.gz")]

# Sort volumes by size
volumes_sorted = sorted(volumes, key=lambda f: f.stat().st_size)

# Take the smallest 2000
smallest_2k = volumes_sorted[:2000]

# Move them
for vol in smallest_2k:
    shutil.move(str(vol), DEST_DIR / vol.name)

print(f"Moved {len(smallest_2k)} smallest volumes to {DEST_DIR}")
